In [ ]:
from pathlib import Path
import sys
from typing import Any, Dict, List, Optional

sys.path.insert(0, str(Path.cwd().parent))

from apis import ticketmaster as tm
from pydantic import BaseModel
import pandas as pd

In [ ]:
class SubField(BaseModel):
    id: str

class Classification(BaseModel):
    primary: Optional[bool] = None
    segment: Optional[SubField] = None
    genre: Optional[SubField] = None
    subGenre: Optional[SubField] = None
    type: Optional[SubField] = None
    subType: Optional[SubField] = None
    family: Optional[bool] = None

class TicketingFeature(BaseModel):
    enabled: bool

class Ticketing(BaseModel):
    safeTix: Optional[TicketingFeature] = None
    allInclusivePricing: Optional[TicketingFeature] = None

class AgeRestrictions(BaseModel):
    legalAgeEnforced: bool

class TicketTextLinesLanguage(BaseModel):
    line1: Optional[str] = None
    line2: Optional[str] = None
    line3: Optional[str] = None
    line4: Optional[str] = None
    line5: Optional[str] = None
    line6: Optional[str] = None

class LinksHref(BaseModel):
    href: str

class Links(BaseModel):
    self: LinksHref
    attractions: Optional[List[LinksHref]] = None
    venues: Optional[List[LinksHref]] = None

class Location(BaseModel):
    longitude: str
    latitude: str

class Venue(BaseModel):
    type: str
    id: str
    test: bool
    url: str
    locale: str
    images: List[Dict[str, Any]]
    postalCode: Optional[str] = None
    timezone: Optional[str] = None
    city: Optional[Dict[str, Any]] = None
    state: Optional[Dict[str, Any]] = None
    country: Optional[Dict[str, Any]] = None
    address: Optional[Dict[str, Any]] = None
    location: Optional[Location] = None
    markets: Optional[List[SubField]] = None
    dmas: Optional[List[Dict[str, Any]]] = None
    upcomingEvents: Optional[Dict[str, Any]] = None
    ada: Optional[Dict[str, Any]] = None
    _links: Optional[Links] = None

class Attraction(BaseModel):
    name: str
    type: str
    id: str
    test: bool
    url: str
    locale: str
    externalLinks: Optional[Dict[str, Any]] = None
    images: List[Dict[str, Any]] = None
    classifications: List[Classification] = None
    upcomingEvents: Optional[Dict[str, Any]] = None
    _links: Optional[Links] = None

class Embedded(BaseModel):
    venues: Optional[List[Venue]] = None
    attractions: Optional[List[Attraction]] = None

class Sale(BaseModel):
    startDateTime: str
    startTBD: bool
    startTBA: bool
    endDateTime: str

class Sales(BaseModel):
    public: Sale

class StartDate(BaseModel):
    localDate: str
    localTime: Optional[str] = None
    dateTime: Optional[str] = None
    dateTBD: bool
    dateTBA: bool
    timeTBA: bool
    noSpecificTime: bool

class Status(BaseModel):
    code: str

class Dates(BaseModel):
    start: StartDate
    timezone: str
    status: Status
    spanMultipleDays: bool

class Image(BaseModel):
    ratio: Optional[str] = None
    url: str
    width: Optional[int] = None
    height: Optional[int] = None
    fallback: Optional[bool] = None

class EventLinks(BaseModel):
    self: LinksHref
    attractions: Optional[List[LinksHref]] = None
    venues: Optional[List[LinksHref]] = None

class TMEvent(BaseModel):
    name: str
    type: str
    id: str
    test: bool
    url: str
    locale: str
    images: List[Image]
    sales: Sales
    dates: Dates
    classifications: List[Classification]
    promoter: Optional[SubField] = None
    promoters: Optional[List[SubField]] = None
    ageRestrictions: Optional[AgeRestrictions] = None
    ticketing: Optional[Ticketing] = None
    nameOrigin: Optional[str] = None
    ticketTextLines: Optional[Dict[str, TicketTextLinesLanguage]] = None
    _links: Optional[EventLinks] = None
    _embedded: Optional[Embedded] = None

In [ ]:
endpoint = "/events.json"
kword_filters_gb= {"keyword":"festival", "countryCode":"GB"}
kword_filters_uk= {"keyword":"festival", "countryCode":"UK"}

response_data_gb = tm.get_ticketmaster_data(endpoint, filters=kword_filters_gb)
response_data_uk = tm.get_ticketmaster_data(endpoint, filters=kword_filters_uk)
print(response_data_gb)
events_gb = response_data_gb.get("_embedded", {}).get("events", [])
events_uk = response_data_uk.get("_embedded", {}).get("events", [])
events_list = events_gb + events_uk
events_list[0].keys()

In [ ]:
new_items = [TMEvent.model_validate(item) for item in events_list]
for item in new_items:
    print(item)

In [ ]:
df_events = pd.DataFrame([item.model_dump() for item in new_items])
df_events.head()